# Deploy Orchestrator Agent to Amazon Bedrock AgentCore Runtime

This tutorial series builds a **3-agent e-commerce shopping assistant**. In this notebook, you deploy the Orchestrator Agent using HTTP protocol for client-facing access, while routing queries to specialist agents via A2A protocol.

**Notebook 3 of 3** in the multi-agent orchestration series.

### What You'll Build

A **Orchestrator Agent** built with Strands Agents and deployed to Amazon Bedrock AgentCore that:
- Routes customer queries to Order Agent and Product Agent via A2A protocol
- Discovers specialist capabilities via Agent Cards using `A2AClientToolProvider`
- Authenticates inter-agent calls with AWS SigV4
- Streams responses for real-time progress updates

### How You'll Build It

Using Strands Agents SDK with A2AClientToolProvider and Amazon Bedrock AgentCore:
- Create orchestrator agent with dynamic tool discovery from Agent Cards
- Use `BedrockAgentCoreApp` for HTTP protocol deployment
- Configure SigV4 authentication for secure A2A calls
- Deploy to AgentCore runtime with HTTP protocol (port 8080)

## Architecture Overview

```
User Request
     │
     ▼
┌──────────────────┐
│   Orchestrator   │  ◄── You are here (Notebook 3)
│ (HTTP protocol)  │
└────────┬─────────┘
      │ A2A Protocol (SigV4 Auth)
      ├───────────────────────────────┬
      ▼                               ▼
┌─────────────┐                 ┌─────────────┐
│ Order Agent │                 │Product Agent│
│ (A2A Server)│                 │(A2A Server) │
└─────┬───────┘                 └─────┬───────┘
      │                               │
      ▼                               ▼
┌─────────────┐                 ┌─────────────┐
│  DynamoDB   │                 │Product      │
│  (Orders)   │                 │Catalog API  │
└─────────────┘                 └─────────────┘
```

## Prerequisites

- [AWS CLI](https://aws.amazon.com/cli/) installed and configured
- Python 3.10 or higher
- Docker or Podman installed (only for local container builds; not required if using CodeBuild)
- Claude Sonnet 4 model access in Amazon Bedrock
- **Notebook 01 completed**: Product Agent deployed to Amazon Bedrock AgentCore
- **Notebook 02 completed**: Order Agent deployed to Amazon Bedrock AgentCore
- IAM permissions to:
  - Create IAM roles and policies
  - Create Amazon ECR repositories
  - Deploy Amazon Bedrock AgentCore runtimes
  - Read from AWS Systems Manager Parameter Store
- Local notebook dependencies: `pip install -r requirements.txt`

In [ ]:
# Install notebook dependencies
!pip install -r requirements.txt

In [ ]:
# Import standard libraries
import os
import shutil
from pathlib import Path
from urllib.parse import quote
from uuid import uuid4

# Import AWS SDK
import boto3

# Import utilities
from utils import (
    ORCHESTRATOR_AGENT_NAME,
    ORCHESTRATOR_ROLE_NAME,
    create_agentcore_role,
)

# Get AWS account information
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]
region = "us-west-2"  # Multi-agent tutorial uses us-west-2

# Set paths
NOTEBOOK_DIR = Path.cwd()

# Ensure orchestrator directory exists for %%writefile
(NOTEBOOK_DIR / "orchestrator").mkdir(exist_ok=True)

print(f"AWS Account: {account_id}")
print(f"Region: {region}")
print(f"Notebook Directory: {NOTEBOOK_DIR}")

## Step 1: Create Orchestrator Agent

The Orchestrator Agent is the **client-facing entry point** that routes customer queries to specialist agents. Unlike the Product and Order agents that use A2A protocol for inter-agent communication, the Orchestrator uses HTTP protocol for direct client access with streaming support.

**Key differences from specialist agents:**

| Aspect | Orchestrator | Specialists (Product/Order) |
|--------|-------------|-----------------------------|
| Protocol | HTTP (port 8080) | A2A (port 9000) |
| Entry point | `BedrockAgentCoreApp` | `A2AServer` |
| Client | External users/apps | Other agents |
| Discovery | Discovers specialists via Agent Cards | Advertises via Agent Card |

**How A2AClientToolProvider works:**
1. Fetches Agent Cards from specialist URLs (`/.well-known/agent-card.json`)
2. Parses agent capabilities and input modes
3. Generates callable tools (one per specialist)
4. Model decides which tool to call based on query content

### Code Structure

The orchestrator consists of three files:

| File | What It Does |
|------|--------------|
| `agent.py` | Creates Strands agent with `A2AClientToolProvider` for dynamic specialist discovery and SigV4 authentication |
| `app.py` | `BedrockAgentCoreApp` HTTP server with streaming event processing and structured response filtering |
| `requirements.txt` | Python dependencies including `strands-agents[a2a,otel]` and `bedrock-agentcore` |

In [ ]:
%%writefile orchestrator/agent.py
"""Orchestrator Agent for e-commerce multi-agent orchestration.

Orchestrates Order Agent and Product Agent via A2A protocol to handle customer
queries about orders and products. Routes requests to appropriate specialist agents
and combines responses for unified customer experience.

Key concepts demonstrated:
- A2AClientToolProvider for dynamic agent discovery via Agent Cards
- AWS SigV4 authentication for secure A2A calls to AgentCore
- Streaming callback handler for real-time progress updates
- Customer context propagation via tool_context
"""

import logging
from typing import Any

import boto3
from strands import Agent, tool
from strands.models import BedrockModel
from strands.types.tools import ToolContext
from strands_tools.a2a_client import A2AClientToolProvider

from sigv4_auth import SigV4HTTPXAuth

logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("strands").setLevel(logging.INFO)
logger = logging.getLogger(__name__)

# --- Callback Handler ---
class StreamingProgressHandler:
    """Log A2A routing decisions and tool results for debugging multi-agent workflows.
    
    Tracks which tools the agent calls and with what inputs. Useful for:
    - Debugging routing decisions between Order and Product agents
    - Monitoring A2A calls in distributed systems
    - Building observability dashboards
    """

    def __init__(self) -> None:
        self.progress_shown_ids: set[str] = set()  # Prevent duplicate progress messages
        self.logged_tool_ids: set[str] = set()     # Prevent duplicate logs

    def __call__(self, **kwargs: Any) -> None:
        # Show progress immediately when tool name arrives
        current_tool_use = kwargs.get("current_tool_use", {})
        if current_tool_use and current_tool_use.get("name"):
            tool_id = current_tool_use.get("toolUseId")
            tool_name = current_tool_use.get("name", "")

            if tool_id and tool_id not in self.progress_shown_ids:
                self.progress_shown_ids.add(tool_id)

                if "a2a_send_message" in tool_name:
                    print("\n[Calling agent...]", flush=True)
                elif "a2a_list" in tool_name:
                    print("\n[Discovering available agents...]", flush=True)

        # Log routing decisions after tool input is complete
        message = kwargs.get("message", {})
        if not isinstance(message, dict):
            return

        if message.get("role") == "assistant":
            for content in message.get("content", []):
                if isinstance(content, dict):
                    tool_use = content.get("toolUse")
                    if tool_use:
                        tool_id = tool_use.get("toolUseId")
                        if tool_id and tool_id not in self.logged_tool_ids:
                            self.logged_tool_ids.add(tool_id)
                            tool_name = tool_use.get("name", "")
                            tool_input = tool_use.get("input", {})

                            if "a2a_send_message" in tool_name:
                                input_str = str(tool_input).lower()
                                if "order" in input_str:
                                    logger.info("Routing to Order Agent")
                                elif "product" in input_str:
                                    logger.info("Routing to Product Agent")

        # Handle tool results
        elif message.get("role") == "user":
            for content in message.get("content", []):
                if isinstance(content, dict) and "toolResult" in content:
                    status = content["toolResult"].get("status", "")
                    if status == "success":
                        print("[Agent response received]", flush=True)
                    elif status == "error":
                        print("[Agent error]", flush=True)


# --- Configuration ---

SYSTEM_PROMPT = """You are the E-commerce Shopping Assistant Orchestrator.

Route customer queries to the Order Agent (order status, history) or Product Agent (catalog, browsing).
For queries needing both, call Order Agent first to get product_ids, then Product Agent for details.
Respond in a friendly, personalized way."""

# --- Tools ---


@tool(context=True)
def get_customer_context(tool_context: ToolContext) -> dict:
    """Retrieve current customer context including customer ID from session state.

    Enables orchestrator to access session-level state (customer_id) set during
    invocation. Specialist agents use this context to filter results appropriately.

    Args:
        tool_context: Strands Agents ToolContext containing invocation_state dictionary.

    Returns:
        Dictionary with customer_id from session state or "unknown" as fallback.
    """
    return {"customer_id": tool_context.invocation_state.get("customer_id", "unknown")}


# --- Agent Creation ---


def create_orchestrator(order_agent_url: str, product_agent_url: str) -> Agent:
    """Create orchestrator agent with A2A client tools for multi-agent orchestration.

    Sets up A2AClientToolProvider to communicate with specialist agents deployed to
    Amazon Bedrock AgentCore. The provider fetches Agent Cards from each specialist
    URL to discover their capabilities and generates callable tools for the orchestrator.

    Args:
        order_agent_url: AgentCore invocation URL for Order Agent.
        product_agent_url: AgentCore invocation URL for Product Agent.

    Returns:
        Configured Agent instance ready to orchestrate multi-agent workflows.
    """
    logger.info(f"Initializing A2A client for Order Agent: {order_agent_url}")
    logger.info(f"Initializing A2A client for Product Agent: {product_agent_url}")

    session = boto3.Session()
    region = session.region_name or "us-west-2"
    credentials = session.get_credentials()

    logger.info(f"Using AWS region: {region}")

    # Configure SigV4 authentication for A2A protocol calls to AgentCore
    auth = SigV4HTTPXAuth(
        credentials=credentials,
        service="bedrock-agentcore",
        region=region,
    )

    logger.info("AWS SigV4 authentication configured for A2A protocol")

    # A2AClientToolProvider discovers agent capabilities from Agent Cards
    # and generates callable tools (one per specialist agent)
    a2a_provider = A2AClientToolProvider(
        known_agent_urls=[order_agent_url, product_agent_url],
        httpx_client_args={"auth": auth},
    )

    logger.info(f"A2A client tools available: {len(a2a_provider.tools)} tools")

    agent = Agent(
        name="Ecommerce_Shopping_Orchestrator",
        description="Coordinates customer shopping queries using Order and Product agents",
        system_prompt=SYSTEM_PROMPT,
        model=BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0", region_name=region),
        tools=[get_customer_context] + a2a_provider.tools,
        callback_handler=StreamingProgressHandler(),
    )

    logger.info(f"Agent created: {agent.name}")

    return agent

In [ ]:
%%writefile orchestrator/app.py
"""Orchestrator application deployed to Amazon Bedrock AgentCore with HTTP protocol.

Multi-agent orchestrator that routes customer queries to Order and Product specialist
agents via A2A protocol. Uses BedrockAgentCoreApp for HTTP protocol deployment with
streaming event processing.

Key concepts demonstrated:
- BedrockAgentCoreApp for HTTP protocol integration (port 8080)
- OpenTelemetry instrumentation for AWS X-Ray distributed tracing
- AWS Systems Manager Parameter Store for agent URL configuration
- Structured event filtering for streaming visualization
"""

import logging
import os
from datetime import datetime

import boto3
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.telemetry import StrandsTelemetry

from agent import create_orchestrator

# --- Logging ---

logging.basicConfig(level=logging.INFO)
logging.getLogger("strands").setLevel(logging.INFO)
logger = logging.getLogger(__name__)

# Enable distributed tracing with AWS X-Ray
StrandsTelemetry().setup_otlp_exporter()

# --- Configuration ---
SSM_ORDER_AGENT_URL = "/ecommerce-assistant/order-agent-url"
SSM_PRODUCT_AGENT_URL = "/ecommerce-assistant/product-agent-url"

logger.info("=" * 60)
logger.info("=== Shopping Assistant Orchestrator Initialization ===")
logger.info("=" * 60)

# --- Agent Runtime Initialization ---
app = BedrockAgentCoreApp()


def get_agent_url(ssm_param: str, env_var: str) -> str:
    """Get agent URL from environment variable or SSM Parameter Store.

    Checks environment variable first (useful for local dev), then falls back
    to SSM (standard for AgentCore deployments where config is stored centrally).
    """
    if url := os.environ.get(env_var):
        logger.info(f"Using {env_var} from environment: {url}")
        return url
    logger.info(f"Reading URL from SSM: {ssm_param}")
    ssm = boto3.client("ssm")
    response = ssm.get_parameter(Name=ssm_param, WithDecryption=True)
    return response["Parameter"]["Value"]


# Initialize orchestrator at module level (created once on container start)
logger.info("Retrieving agent URLs from SSM...")
order_agent_url = get_agent_url(SSM_ORDER_AGENT_URL, "ORDER_AGENT_URL")
product_agent_url = get_agent_url(SSM_PRODUCT_AGENT_URL, "PRODUCT_AGENT_URL")
logger.info(f"Order Agent URL: {order_agent_url}")
logger.info(f"Product Agent URL: {product_agent_url}")

logger.info("Creating orchestrator with A2A client...")
orchestrator = create_orchestrator(order_agent_url, product_agent_url)
logger.info(f"Orchestrator created: {orchestrator.name}")
logger.info("=" * 60)
logger.info("=== Orchestrator Initialization Complete ===")
logger.info("=" * 60)

# --- Entry Point ---
@app.entrypoint
async def invoke(payload):
    """Handle incoming requests with streaming response for multi-agent orchestration.

    Entry point called by AgentCore when request arrives at runtime endpoint.
    Async streaming provides real-time progress updates as orchestrator routes
    queries to specialist agents.

    Args:
        payload: Request payload containing prompt and optional customer_id.

    Yields:
        Structured event dictionaries for client-side rendering.
    """
    user_input = payload.get("prompt", "")
    customer_id = payload.get("customer_id", "CUST-101")

    logger.info("=" * 60)
    logger.info("=== INCOMING REQUEST ===")
    logger.info(f"Timestamp: {datetime.now().isoformat()}")
    logger.info(f"Customer: {customer_id}")
    logger.info(f"Prompt: {user_input}")
    logger.info("=" * 60)

    logger.info("Starting streaming response...")
    async for event in orchestrator.stream_async(user_input, customer_id=customer_id):
        if isinstance(event, dict):
            # Strands Agents callback events
            if event.get("start_event_loop"):
                yield {"type": "thinking"}

            if "message" in event:
                msg = event["message"]
                if isinstance(msg, dict) and msg.get("role") == "user":
                    for content in msg.get("content", []):
                        if isinstance(content, dict) and "toolResult" in content:
                            status = content["toolResult"].get("status", "success")
                            yield {"type": "tool_result", "status": status}

            if "event_loop_throttled_delay" in event:
                yield {"type": "throttled", "seconds": event["event_loop_throttled_delay"]}

            if "reasoningText" in event:
                yield {"type": "reasoning", "content": event["reasoningText"]}

            # Amazon Bedrock streaming events
            if "event" in event:
                inner = event.get("event", {})

                if "messageStart" in inner:
                    yield {"type": "message_start"}

                elif "contentBlockStart" in inner:
                    start = inner["contentBlockStart"]
                    if "toolUse" in start.get("start", {}):
                        tool_info = start["start"]["toolUse"]
                        yield {
                            "type": "tool_start",
                            "name": tool_info.get("name", "unknown"),
                            "id": tool_info.get("toolUseId", ""),
                        }

                elif "contentBlockDelta" in inner:
                    delta = inner["contentBlockDelta"].get("delta", {})

                    if "text" in delta:
                        yield {"type": "text", "content": delta["text"]}

                    if "toolUse" in delta:
                        tool_input = delta["toolUse"].get("input", "")
                        yield {"type": "tool_input", "input": tool_input}

                elif "contentBlockStop" in inner:
                    yield {"type": "tool_complete"}

                elif "messageStop" in inner:
                    stop_reason = inner["messageStop"].get("stopReason", "")
                    yield {"type": "message_stop", "reason": stop_reason}

        elif isinstance(event, str):
            yield {"type": "text", "content": event}

    logger.info("=" * 60)
    logger.info("=== STREAMING COMPLETE ===")
    logger.info("=" * 60)


if __name__ == "__main__":
    app.run()

In [ ]:
%%writefile orchestrator/requirements.txt
strands-agents[a2a,otel]==1.29.0
strands-agents-tools==0.2.22
bedrock-agentcore==1.4.2
fastapi==0.115.0
uvicorn==0.34.0
boto3==1.42.60

### SigV4 Authentication for A2A Calls

When the orchestrator invokes specialist agents on Amazon Bedrock AgentCore, those calls are AWS API requests that require **AWS Signature Version 4 (SigV4)** authentication.

**What SigV4 does:**
- Creates a cryptographic signature from your request and AWS credentials
- Verifies the caller has `bedrock-agentcore:InvokeAgentRuntime` permission
- Prevents request tampering and replay attacks (5-minute validity window)

**Authentication flow:**

```
Orchestrator Agent                    AgentCore Runtime               A2A Agent Container
       |                                    |                               |
       | 1. Build A2A request               |                               |
       |                                    |                               |
       | 2. SigV4HTTPXAuth signs request    |                               |
       |----------------------------------->|                               |
       |                                    | 3. Verify signature           |
       |                                    | 4. Route to container         |
       |                                    |------------------------------>|
       |                                    |                               | 5. Process
       |                                    |<------------------------------|
       |<-----------------------------------|                               |
       | 6. Response                        |                               |
```

The `sigv4_auth.py` utility provides `SigV4HTTPXAuth`, which automatically signs HTTP requests using your AWS credentials. Copy this file to the orchestrator directory so it's included in the container.

In [ ]:
# Copy sigv4_auth.py to orchestrator directory
sigv4_source = NOTEBOOK_DIR / "utils" / "sigv4_auth.py"
sigv4_dest = NOTEBOOK_DIR / "orchestrator" / "sigv4_auth.py"

shutil.copy(sigv4_source, sigv4_dest)
print(f"Copied sigv4_auth.py to orchestrator directory")

---
## Step 2: Deploy to Amazon Bedrock AgentCore

The `bedrock-agentcore-starter-toolkit` handles the deployment pipeline. Behind the scenes, it:

1. **Creates IAM role** — Grants permissions for ECR image pull, Bedrock model invocation, SSM read access, and CloudWatch logging
2. **Configures runtime** — Packages agent code with dependencies and sets HTTP protocol
3. **Builds container** — Creates Docker image and pushes to Amazon ECR
4. **Launches runtime** — Starts managed container in Amazon Bedrock AgentCore

### What Gets Created

| Resource | Name/Path | Purpose |
|----------|-----------|---------|  
| IAM Role | `ecommerce-orchestrator-role` | Grants the agent runtime permissions to pull container images from Amazon ECR, invoke Amazon Bedrock models, read SSM parameters, and write logs to Amazon CloudWatch |
| ECR Repository | `ecommerce_orchestrator` | Stores the Docker container image that Amazon Bedrock AgentCore pulls to run the agent |
| AgentCore Runtime | `ecommerce_orchestrator` | Managed compute environment that runs your containerized HTTP server and handles scaling, networking, and health monitoring |

### Create IAM Role and Configure Runtime

The IAM execution role allows the AgentCore runtime to pull container images, invoke Bedrock models, read SSM parameters, and write logs.

| Parameter | Value | Purpose |
|-----------|-------|---------|  
| `entrypoint` | `app.py` | Python file that starts the BedrockAgentCoreApp HTTP server |
| `protocol` | `HTTP` | Enables port 8080 for client-facing requests with streaming support |
| `auto_create_ecr` | `True` | Creates ECR repository if it doesn't exist |

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

# Create IAM execution role for Orchestrator
orchestrator_role_arn = create_agentcore_role(ORCHESTRATOR_ROLE_NAME, account_id, region)
print(f"IAM Role ARN: {orchestrator_role_arn}")

# Change to orchestrator directory
orchestrator_dir = NOTEBOOK_DIR / "orchestrator"
os.chdir(orchestrator_dir)

# Configure runtime
orchestrator_runtime = Runtime()
orchestrator_runtime.configure(
    entrypoint="app.py",
    execution_role=orchestrator_role_arn,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=ORCHESTRATOR_AGENT_NAME,
    protocol="HTTP",  # HTTP for client-facing orchestrator (not A2A)
)
print("Orchestrator runtime configured with HTTP protocol")

### Launch Agent

`Runtime.launch()` builds the Docker image, pushes it to ECR, and creates the AgentCore runtime. The `auto_update_on_conflict=True` flag updates an existing runtime if one already exists with the same name.

**Note:** First deployment takes 5-10 minutes (subsequent updates are faster).

In [ ]:
# Launch Orchestrator
print("Launching Orchestrator (this may take several minutes)...")
orchestrator_launch = orchestrator_runtime.launch(auto_update_on_conflict=True)
print(f"Orchestrator ARN: {orchestrator_launch.agent_arn}")
ORCHESTRATOR_ARN = orchestrator_launch.agent_arn

### Get Runtime URL

Once the runtime reaches `ACTIVE` or `READY` status, construct the invocation URL from the runtime ARN. This URL is the HTTPS endpoint for client applications to send queries to the orchestrator.

In [ ]:
# Get runtime status and URL
status_response = orchestrator_runtime.status()
status = status_response.endpoint.get("status", "")
orchestrator_url = None

print(f"Orchestrator Status: {status}")

if status.upper() in ["ACTIVE", "READY"]:
    agent_runtime_arn = status_response.endpoint.get("agentRuntimeArn")
    escaped_arn = quote(agent_runtime_arn, safe="")
    orchestrator_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_arn}/invocations"
    print(f"Orchestrator URL: {orchestrator_url}")
else:
    print(f"WARNING: Orchestrator status is {status}")
    print("Expected status: ACTIVE or READY")

In [ ]:
# Return to notebook directory
os.chdir(NOTEBOOK_DIR)

print("=" * 60)
print("Orchestrator Deployment Summary")
print("=" * 60)
print(f"Agent Name: {ORCHESTRATOR_AGENT_NAME}")
print(f"Agent ARN: {ORCHESTRATOR_ARN}")
print(f"IAM Role: {orchestrator_role_arn}")
print(f"Runtime URL: {orchestrator_url or 'Not available - agent not ready'}")
print(f"Status: {status}")
print(f"Region: {region}")
print(f"Protocol: HTTP (port 8080)")
print("=" * 60)

---
## Step 3: Test Multi-Agent System

Verify the orchestrator routes queries correctly to specialist agents. The tests below demonstrate:

| Scenario | Expected Routing | What It Tests |
|----------|------------------|---------------|
| Product query | Product Agent only | Catalog search via DummyJSON API |
| Order query | Order Agent only | Order lookup via DynamoDB |
| Combined query | Both agents | Multi-agent orchestration |

The `StreamingDisplay` utility visualizes A2A protocol calls in real-time, showing which agent handles each request.

In [ ]:
import json
from utils.streaming import StreamingDisplay

# Create AgentCore client for streaming invocation
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)


def test_orchestrator(prompt: str, customer_id: str = "CUST-101"):
    """Run a test query through the orchestrator with streaming visualization."""
    # Build the invocation payload with user prompt and customer context
    payload = {"prompt": prompt, "customer_id": customer_id}

    # Initialize streaming display for real-time response visualization
    display = StreamingDisplay(customer_id=customer_id, query=prompt)
    display.start()

    try:
        # Invoke the Amazon Bedrock AgentCore runtime with streaming enabled
        response = agentcore_client.invoke_agent_runtime(
            agentRuntimeArn=ORCHESTRATOR_ARN,  # ARN of the deployed agent runtime
            qualifier="DEFAULT",               # Runtime version qualifier
            payload=json.dumps(payload)
        )

        # Check if response is a SSE stream
        if "text/event-stream" in response.get("contentType", ""):
            # Process SSE stream line by line as events arrive
            for line in response["response"].iter_lines(chunk_size=1):
                if line:
                    line = line.decode("utf-8")
                    # SSE data lines are prefixed with "data: "
                    if line.startswith("data: "):
                        data = line[6:]  # Strip SSE prefix
                        try:
                            # Parse JSON event from stream
                            event = json.loads(data)
                            if isinstance(event, dict):
                                # Structured event: delegate to display handler
                                display.handle_event(event)
                            else:
                                # Plain text: output directly
                                print(event, end="", flush=True)
                        except json.JSONDecodeError:
                            # Non-JSON data: output as raw text
                            print(data, end="", flush=True)
    except Exception as e:
        print(f"\nError: {e}")

    # Return the accumulated response from all streamed events
    return display.get_full_response()

### Test 1: Product Search

Query about products routes to the **Product Agent**, which searches the DummyJSON catalog API.

In [ ]:
response = test_orchestrator("Show me laptops under $1500")

### Test 2: Order Status

Query about orders routes to the **Order Agent**, which queries DynamoDB for customer order history.

In [ ]:
response = test_orchestrator("What's the status of my recent orders?")

### Test 3: Combined Query

Complex queries may route to **both agents**. The orchestrator calls Order Agent first to get product IDs from order history, then Product Agent for product details.

In [ ]:
response = test_orchestrator("What products have I ordered and are there similar items available?")

---
## Cleanup

Run this section to delete all resources created by this notebook.

In [ ]:
# Destroy Orchestrator
print("Destroying Orchestrator...")
os.chdir(orchestrator_dir)
try:
    orchestrator_runtime.destroy(delete_ecr_repo=True)
    print("Orchestrator destroyed")
except Exception as e:
    print(f"Error: {e}")
os.chdir(NOTEBOOK_DIR)